In [1]:
!pip install kfp[kubernetes]

  Preparing metadata (setup.py) ... done
  DEPRECATION: Building 'kfp-kubernetes' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'kfp-kubernetes'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  Created wheel for kfp-kubernetes: filename=kfp_kubernetes-1.5.0-py3-none-any.whl size=22698 sha256=144e200c5fbff4eee4d5d37cfc79eba94737fc5ea4f67fa21d48634493a99df1
  Stored in directory: /home/jovyan/.cache/pip/wheels/b0/5b/5a/6bf944c532629df9c37498c8b69cdf319e8efd47b1a0e8d5c8
Successfully built kfp-kubernetes
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.30.2
    Uninstalling protobuf-6.30.2:
      Successfully uninstalled protobuf-6.30.2
   ━━━━━━━

In [6]:
import kfp 

from kfp.dsl import component, pipeline
from kfp import kubernetes

In [16]:
@component(
    base_image="ghcr.io/canonical/charmed-spark:3.4-22.04_edge", packages_to_install=["pandas"]
)
def spark_test_component() -> None:
    import logging
    import os
    import pyspark
    import socket
    from lightkube import Client
    from operator import add
    from spark8t.services import K8sServiceAccountRegistry
    from spark8t.services import LightKube as LightKubeInterface
    import pandas    

    app_name = "SparkJob"
    SPARK_SERVICE_ACCOUNT = "kf-spark-user"
    SPARK_NAMESPACE = "admin"

    pod_ip = socket.gethostbyname(socket.gethostname())
    k8s_master = Client().config.cluster.server
    interface = LightKubeInterface(None, None)
    registry = K8sServiceAccountRegistry(interface)
    
    spark_properties = registry.get(
        f"{SPARK_NAMESPACE}:{SPARK_SERVICE_ACCOUNT}"
    ).configurations.props | {
        "spark.driver.host": pod_ip,
        "spark.driver.port": "37371",
        "spark.blockManager.port": "6060",
        "spark.kubernetes.executor.annotation.traffic.sidecar.istio.io/excludeInboundPorts": "37371,6060",
        "spark.kubernetes.executor.annotation.traffic.sidecar.istio.io/excludeOutboundPorts": "37371,6060"
    }

    builder = pyspark.sql.SparkSession\
                    .builder\
                    .appName(app_name).enableHiveSupport()\
                    .master(f"k8s://{k8s_master}")
    for conf, val in spark_properties.items():
        builder = builder.config(conf, val)
    session = builder.getOrCreate()

    df = session.sql("SELECT * FROM default.relation25")

    pdf = df.groupby("window").sum().toPandas()
    
    logging.warning(f"The average messages received in a time window is: {str(pdf['sum(count)'].mean())}")

In [17]:
@pipeline(name="spark-test-pipeline")
def spark_pipeline():
    task = spark_test_component()
    kubernetes.add_pod_label(
        task,
        label_key='access-spark-pipeline',
        label_value='true',
    )
    kubernetes.add_pod_annotation(
        task,
        annotation_key='traffic.sidecar.istio.io/excludeInboundPorts',
        annotation_value='37371,6060',
    )
    kubernetes.add_pod_annotation(
        task,
        annotation_key='traffic.sidecar.istio.io/excludeOutboundPorts',
        annotation_value='37371,6060',
    )

In [18]:
client=kfp.Client()
kfp.compiler.Compiler().compile(
    spark_pipeline,
    package_path="spark_test_pipeline.yaml"
)
run = client.create_run_from_pipeline_func(
    spark_pipeline,
    arguments={},
    enable_caching=False
)